# Phase 4: 50-Epoch Convergence Benchmark (CIFAR-10)
**Project:** Mitigating Clipping Bias in DP-SGD via Error Feedback (Dice-SGD)  
**Authors:** Priyanshu Agarwal & Sudiksha Singh

### Objective
This notebook executes the final, rigorous 50-epoch benchmark comparing baseline DP-SGD against our custom Dice-SGD implementation. To ensure a strictly fair, apples-to-apples comparison, both models are constrained to the optimal hyperparameters discovered in our Phase 3 grid search ($C=1.0, \sigma=0.5, \eta=0.01$).

**Engineering Features:**
1. **In-Place Operation Patching:** We dynamically patch the `forward` method of ResNet's `BasicBlock` to remove in-place memory operations, fulfilling Opacus's strict computational graph requirements.
2. **Robust Checkpointing:** 50-epoch training on a T4 GPU takes significant time. This script implements a "Smart Resume" feature that saves state dictionaries to Google Drive after every epoch, protecting the training run from Colab timeouts.

In [ ]:
# Install required libraries
!pip install -q opacus torchvision

### 1. Environment Setup & Parameter Locking
We lock the hyperparameters to the optimal regime discovered in our grid search. We also mount Google Drive to enable persistent model checkpointing.

In [ ]:
import os
import types
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision.models import resnet18
from torchvision.models.resnet import BasicBlock
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from opacus.optimizers import DPOptimizer
from google.colab import drive

# Mount Drive for robust checkpointing
print("Mounting Google Drive for robust checkpointing...")
drive.mount('/content/drive')

baseline_ckpt_path = '/content/drive/MyDrive/baseline_50_epoch_ckpt.pth'
dice_ckpt_path = '/content/drive/MyDrive/dice_50_epoch_ckpt.pth'
final_img_path = '/content/drive/MyDrive/final_50_epoch_convergence.png'

# --- THE WINNING PARAMETERS ---
WINNING_C = 1.0
WINNING_SIGMA = 0.5
WINNING_LR = 0.01
FINAL_EPOCHS = 50

print(f"\n=== STARTING ROBUST 50-EPOCH BENCHMARK ===")
print(f"Parameters locked: C={WINNING_C}, Sigma={WINNING_SIGMA}, LR={WINNING_LR}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

### 2. Dataset & Architecture Refactoring
We load CIFAR-10 with `num_workers=0` to prevent deadlocks. We also define the `patched_forward` function to rewrite ResNet-18's internal memory management, making it 100% compliant with the Opacus privacy engine.

In [ ]:
print("Downloading CIFAR-10 Dataset & Setting up Dataloader...")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=0)

def patched_forward(self, x):
    """Custom forward pass for ResNet BasicBlock to remove in-place operations."""
    identity = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    if self.downsample is not None:
        identity = self.downsample(x)
    out = out + identity
    out = self.relu(out)
    return out

def get_fresh_model():
    """Generates a fresh, Opacus-compliant ResNet-18."""
    model = resnet18(num_classes=10)
    model = ModuleValidator.fix(model) # Swaps BatchNorm for GroupNorm
    for module in model.modules():
        if hasattr(module, 'inplace'):
            module.inplace = False # Disables in-place memory modifications
        if isinstance(module, BasicBlock):
            module.forward = types.MethodType(patched_forward, module)
    return model.to(device)

### 3. The Custom Dice-SGD Optimizer

In [ ]:
class DiceSGDOptimizer(DPOptimizer):
    """Custom Opacus Optimizer implementing Error Feedback (Dice-SGD)."""
    def __init__(self, optimizer, noise_multiplier, max_grad_norm, expected_batch_size):
        super().__init__(
            optimizer=optimizer,
            noise_multiplier=noise_multiplier,
            max_grad_norm=max_grad_norm,
            expected_batch_size=expected_batch_size,
        )

    def step(self, closure=None):
        # 1. Pre-clip Error Injection
        for group in self.original_optimizer.param_groups:
            for p in group['params']:
                if hasattr(p, 'grad_sample') and p.grad_sample is not None:
                    state = self.original_optimizer.state[p]
                    if 'error_buffer' not in state:
                        state['error_buffer'] = torch.zeros_like(p)
                    p.grad_sample.add_(state['error_buffer'])

        # 2. Opacus Native Clipping and Noising
        self.clip_and_accumulate()
        if self._check_skip_next_step():
            self._is_last_step_skipped = True
            return False
        self.add_noise()
        self.scale_grad()

        # 3. Calculate new clipping bias
        for group in self.original_optimizer.param_groups:
            for p in group['params']:
                if p.grad is not None and hasattr(p, 'grad_sample'):
                    state = self.original_optimizer.state[p]
                    unclipped_mean_grad = torch.mean(p.grad_sample, dim=0)
                    state['error_buffer'] = unclipped_mean_grad - p.grad.detach()

        # 4. Standard step
        self.original_optimizer.step(closure)
        self.zero_grad()
        self._is_last_step_skipped = False
        return True

### 4. Part A: Baseline DP-SGD Training
We execute the baseline run with smart resuming logic. If training is interrupted, rerunning this cell will seamlessly load the weights from Drive and continue.

In [ ]:
print("\n--- Phase 1: Training Baseline DP-SGD ---")
baseline_model = get_fresh_model()
baseline_optimizer = optim.SGD(baseline_model.parameters(), lr=WINNING_LR, momentum=0.5)
baseline_privacy_engine = PrivacyEngine(accountant="rdp")

baseline_model, baseline_optimizer, baseline_trainloader = baseline_privacy_engine.make_private(
    module=baseline_model,
    optimizer=baseline_optimizer,
    data_loader=trainloader,
    noise_multiplier=WINNING_SIGMA,
    max_grad_norm=WINNING_C,
)

baseline_start_epoch = 0
baseline_losses = []

# Smart Resume Logic
if os.path.exists(baseline_ckpt_path):
    print("Found existing Baseline checkpoint! Resuming...")
    checkpoint = torch.load(baseline_ckpt_path)
    baseline_model.load_state_dict(checkpoint['model_state_dict'])
    baseline_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    baseline_losses = checkpoint['losses']
    baseline_start_epoch = checkpoint['epoch']
    print(f"Resuming Baseline from Epoch {baseline_start_epoch + 1}...")

if baseline_start_epoch >= FINAL_EPOCHS:
    print("Baseline DP-SGD is already 100% complete. Skipping to Phase 2.")
else:
    for epoch in range(baseline_start_epoch, FINAL_EPOCHS):
        baseline_model.train()
        running_loss = 0.0
        for data in baseline_trainloader:
            inputs, labels = data[0].to(device), data[1].to(device)
            baseline_optimizer.zero_grad()
            outputs = baseline_model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            baseline_optimizer.step()
            running_loss += loss.item()

        epoch_loss = running_loss / len(baseline_trainloader)
        baseline_losses.append(round(epoch_loss, 3))
        print(f"  Baseline Epoch {epoch + 1}/{FINAL_EPOCHS} | Loss: {epoch_loss:.3f}")

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': baseline_model.state_dict(),
            'optimizer_state_dict': baseline_optimizer.state_dict(),
            'losses': baseline_losses
        }, baseline_ckpt_path)

### 5. Part B: Dice-SGD Training
We execute our custom optimizer on the exact same constraints.

In [ ]:
print("\n--- Phase 2: Training Dice-SGD ---")
dice_model = get_fresh_model()
dice_optimizer = optim.SGD(dice_model.parameters(), lr=WINNING_LR, momentum=0.5)
dice_privacy_engine = PrivacyEngine(accountant="rdp")

dice_model, dice_optimizer, dice_trainloader = dice_privacy_engine.make_private(
    module=dice_model,
    optimizer=dice_optimizer,
    data_loader=trainloader,
    noise_multiplier=WINNING_SIGMA,
    max_grad_norm=WINNING_C,
)

# INJECT THE CUSTOM ENGINE
dice_optimizer.__class__ = DiceSGDOptimizer

dice_start_epoch = 0
dice_losses = []

# Smart Resume Logic
if os.path.exists(dice_ckpt_path):
    print("Found existing Dice-SGD checkpoint! Resuming...")
    checkpoint = torch.load(dice_ckpt_path)
    dice_model.load_state_dict(checkpoint['model_state_dict'])
    dice_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    dice_losses = checkpoint['losses']
    dice_start_epoch = checkpoint['epoch']
    print(f"Resuming Dice-SGD from Epoch {dice_start_epoch + 1}...")

if dice_start_epoch >= FINAL_EPOCHS:
    print("Dice-SGD is already 100% complete.")
else:
    for epoch in range(dice_start_epoch, FINAL_EPOCHS):
        dice_model.train()
        running_loss = 0.0
        for data in dice_trainloader:
            inputs, labels = data[0].to(device), data[1].to(device)
            dice_optimizer.zero_grad()
            outputs = dice_model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            dice_optimizer.step()
            running_loss += loss.item()

        epoch_loss = running_loss / len(dice_trainloader)
        dice_losses.append(round(epoch_loss, 3))
        print(f"  Dice-SGD Epoch {epoch + 1}/{FINAL_EPOCHS} | Loss: {epoch_loss:.3f}")

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': dice_model.state_dict(),
            'optimizer_state_dict': dice_optimizer.state_dict(),
            'losses': dice_losses
        }, dice_ckpt_path)

### 6. Part C: The Final Hero Graph
Generating the comparative visualization showing the loss drop from $\sim1.65$ to $\sim0.007$.

In [ ]:
if len(baseline_losses) == FINAL_EPOCHS and len(dice_losses) == FINAL_EPOCHS:
    print("\n=== TRAINING COMPLETE. GENERATING FINAL GRAPH ===")

    epochs_x = list(range(1, FINAL_EPOCHS + 1))
    plt.figure(figsize=(10, 6), dpi=150)

    plt.plot(epochs_x, baseline_losses, marker='', linestyle='--', color='red', linewidth=2, label='Baseline DP-SGD')
    plt.plot(epochs_x, dice_losses, marker='', linestyle='-', color='blue', linewidth=2, label='Dice-SGD (Custom Optimizer)')

    plt.title('50-Epoch Convergence: Baseline DP-SGD vs. Dice-SGD', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('Epochs', fontsize=14)
    plt.ylabel('Cross-Entropy Loss', fontsize=14)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.legend(fontsize=12)
    plt.tight_layout()

    plt.savefig(final_img_path)
    print(f"Final Hero Graph safely saved to Drive: {final_img_path}")
    plt.show()
else:
    print("\nWaiting for both models to finish before graphing...")